# Notebook for LiteRt-LM testing

- Production-ready, open-source inference framework designed to deliver high-performance, cross-platform LLM deployments on edge devices.

## Libraries and settings

In [28]:
import time
import re
import litert_lm

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

MODEL_PATH = "/home/mjrovai/Documents/LITERT/models/gemma-4-E2B-it.litertlm"

## Simple Query

In [29]:
PROMPT = "What is the capital of Brazil"

In [34]:
with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation() as conversation:
        # Synchronous, single-turn call
        response = conversation.send_message(PROMPT)
        # LiteRT-LM docs: response["content"][0]["text"] has the text.[web:139]
        answer = response["content"][0]["text"]
        print(answer)

The capital of Brazil is **Brasília**.


In [32]:
def simple_query(PROMPT, model=MODEL_PATH):
    with litert_lm.Engine(model) as engine:
        with engine.create_conversation() as conversation:
            # Synchronous, single-turn call
            response = conversation.send_message(PROMPT)
            # LiteRT-LM docs: response["content"][0]["text"] has the text.[web:139]
            answer = response["content"][0]["text"]
            print(answer)

In [36]:
PROMPT = "What is the capital of France"
simple_query(PROMPT)

The capital of France is **Paris**.


In [37]:
PROMPT = "And Peru?"
simple_query(PROMPT)

Please provide more context! "And Peru?" is a very open-ended question. To give you a helpful answer, I need to know what you are referring to.

For example, are you asking about:

* **Geography?** (e.g., "And Peru's geography?")
* **History?** (e.g., "And Peru's history?")
* **Culture?** (e.g., "And Peru's culture?")
* **Tourism?** (e.g., "And Peru's top tourist attractions?")
* **A specific topic you were just discussing?** (If so, please remind me!)
* **Something else entirely?**

**Tell me what you'd like to know about Peru!** 😊


> Note that there is "no history". 

## Chat

In [35]:
with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation() as conversation:
        while True:
            user_input = input("\n>>> ").strip()

            # Exit condition
            if user_input.lower() in {"exit", "quit"}:
                print("\n[Exiting chat]")
                break
            if not user_input:
                continue

            for chunk in conversation.send_message_async(user_input):
                print(chunk["content"][0]["text"], end="", flush=True)


>>>  What is teh Capital of France?


The capital of France is **Paris**.


>>>  And Peru?


The capital of Peru is **Lima**.


>>>  /exit


Goodbye! Let me know if you have any other questions.


>>>  exit



[Exiting chat]


>Note that you do not need to manually rebuild message history turn by turn as you often do with raw chat APIs, as long as you stay inside the same `conversation` object.

## Tools

In [19]:
PROMPT = "What is 123456 multiplied by 123456?"
simple_query(PROMPT)

Here are a few ways to approach this problem:

**1. Direct Multiplication (The most straightforward, but tedious):**

You need to multiply the two 6-digit numbers:
$$123,456 \times 123,456$$

This is a large multiplication. We can use the property that $N \times N = N^2$.

**2. Using the Square of a Number (The most efficient method):**

Let $N = 123,456$. We are looking for $N^2$.

We can use the algebraic identity for squaring a number, or simply perform the multiplication.

**Calculation:**

Let's perform the multiplication:
$$123,456 \times 123,456$$

Using a calculator or computational tools, the result is:

$$123,456^2 = 15,238,827,536$$

---
**Verification (Conceptual Check):**

Since the numbers are relatively large, the result will have $6 + 6 = 12$ digits.

*   $120,000 \times 120,000 = 14,400,000,000$
*   The actual result is slightly larger than this.

**Answer:**

$$123,456 \times 123,456 = 15,238,827,536$$


>The answer is wrong!

Let's verify it with simple Python:

In [41]:
print(f"{123456*123456:,}")

15,241,383,936


We can define Python functions as tools that the model can call automatically

### 1. Define the Tool

In [42]:
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers.

    Args:
        a: The first number.
        b: The second number.
    """
    return f'{a * b:,}'

### 2. Define the system prompt

In [43]:
import json

SYSTEM_INSTRUCTION = """
You are a helpful assistant with access to a single tool: multiply_numbers(a, b).

When the user asks for a multiplication, DO NOT explain.
Instead, respond ONLY with a JSON object of this form:

{
  "tool": "multiply_numbers",
  "a": <number>,
  "b": <number>
}

No extra text. No Markdown. No backticks.
If the user is not asking for a multiplication, answer normally in plain text.
"""

### 3. Single "tool call" turn

In [44]:
with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation() as conversation:
        # 1) Prime with the system-style instruction
        conversation.send_message(SYSTEM_INSTRUCTION)

        # 2) User asks a question that should trigger the tool
        user_prompt = "What is 123456 multiplied by 123456?"
        response = conversation.send_message(user_prompt)
        text = response["content"][0]["text"]
        print("Model raw output:\n", text)

        # 3) Try to parse JSON and call the tool
        result = None
        try:
            tool_call = json.loads(text)
            if tool_call.get("tool") == "multiply_numbers":
                a = float(tool_call["a"])
                b = float(tool_call["b"])
                result = multiply_numbers(a, b)
        except Exception as e:
            print("\n[Could not parse tool call or invalid format]", e)

        if result is not None:
            print(f"\n[Tool result] multiply_numbers({a}, {b}) = {result}")
        else:
            print("\n[No tool call detected; treat model reply as plain text]")


Model raw output:
 {
  "tool": "multiply_numbers",
  "a": 123456,
  "b": 123456
}

[Tool result] multiply_numbers(123456.0, 123456.0) = 15,241,383,936.0


In [45]:
with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation() as conversation:
        # 1) Prime with the system-style instruction
        conversation.send_message(SYSTEM_INSTRUCTION)

        # 2) User asks a question that should trigger the tool
        user_prompt = "What is the capital of Guatemala?"
        response = conversation.send_message(user_prompt)
        text = response["content"][0]["text"]
        print("Model raw output:\n", text)

        # 3) Try to parse JSON and call the tool
        result = None
        try:
            tool_call = json.loads(text)
            if tool_call.get("tool") == "multiply_numbers":
                a = float(tool_call["a"])
                b = float(tool_call["b"])
                result = multiply_numbers(a, b)
        except Exception as e:
            print("\n[Could not parse tool call or invalid format]", e)

        if result is not None:
            print(f"\n[Tool result] multiply_numbers({a}, {b}) = {result}")
        else:
            print("\n[No tool call detected; treat model reply as plain text]")


Model raw output:
 The capital of Guatemala is Guatemala City.

[Could not parse tool call or invalid format] Expecting value: line 1 column 1 (char 0)

[No tool call detected; treat model reply as plain text]


This gives the feel of tool calling:
- The model "proposes" a tool invocation in JSON.
- Python verifies and executes it.
- You then optionally pass the result back into the conversation as another turn.

You can later swap `multiply_numbers` for anything else (filesystem check, sensor reading, robot action) while keeping exactly the same pattern.

## Including Metrics and "chunks"

In [ ]:
def rough_token_count(text: str) -> int:
    return len(re.findall(r"\w+|[^\w\s]", text, re.UNICODE))

def chunk_to_text(chunk) -> str:
    parts = []
    if isinstance(chunk, dict):
        for item in chunk.get("content", []):
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
    return "".join(parts)

with litert_lm.Engine(MODEL_PATH) as engine:
    with engine.create_conversation() as conversation:
        while True:
            user_input = input("\n>>> ").strip()

            if user_input.lower() in {"exit", "quit"}:
                break
            if not user_input:
                continue

            start = time.perf_counter()
            first_token_time = None
            pieces = []

            for chunk in conversation.send_message_async(user_input):
                text = chunk_to_text(chunk)
                if text:
                    if first_token_time is None:
                        first_token_time = time.perf_counter()
                    pieces.append(text)
                    print(text, end="", flush=True)

            end = time.perf_counter()
            print()

            output = "".join(pieces)
            output_tokens = rough_token_count(output)
            total_latency = end - start
            ttft = (first_token_time - start) if first_token_time else None
            decode_time = (end - first_token_time) if first_token_time else None
            tps = (output_tokens / decode_time) if decode_time and decode_time > 0 else 0.0

            print(
                f"[TTFT: {ttft:.3f}s | Total: {total_latency:.3f}s | "
                f"OutTok(est): {output_tokens} | Tk/s(est): {tps:.2f}]"
                if ttft is not None
                else f"[Total: {total_latency:.3f}s | OutTok(est): {output_tokens}]"
            )


>>>  what is the capital of Brazil?


The capital of Brazil is **Brasília**.
[TTFT: 1.524s | Total: 3.438s | OutTok(est): 11 | Tk/s(est): 5.75]



>>>  and France?


The capital of France is **Paris**.
[TTFT: 1.468s | Total: 3.156s | OutTok(est): 11 | Tk/s(est): 6.52]



>>>  y la de Bolivia?


The capital of Bolivia is **Sucre**.

However, it's important to note that **La Paz** is the administrative capital and the seat of the government, while **Sucre** is the constitutional capital. Sucre holds the constitutional status.
[TTFT: 1.427s | Total: 11.427s | OutTok(est): 54 | Tk/s(est): 5.40]
